In [ ]:
import mne
import numpy as np

In [ ]:
import numpy as np

In [ ]:
SLEEP_STAGES_INCLUDE = {2}  # matcht de placeholder — alles wordt meegenomen

In [3]:
import mne
import numpy as np

def load_edf(edf_path: str, subject_id: str, night_id: str,
             eeg_channel: str = "EEG",
             emg_channel: str = None,
             accel_channels: tuple = ("X", "Y", "Z")) -> dict:
    """
    Laadt een .edf bestand en zet het om naar het night-dict
    dat run_one_night() verwacht.

    Parameters
    ----------
    edf_path      : pad naar je .edf bestand
    subject_id    : naam/ID van de proefpersoon
    night_id      : naam van de nacht (bv. "night_1")
    eeg_channel   : kanaalnaam in je EDF voor EEG
    emg_channel   : kanaalnaam voor EMG (optioneel)
    accel_channels: kanaalnamen voor x/y/z accelerometer
    """
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    fs  = raw.info["sfreq"]

    print(f"Kanalen: {raw.ch_names}")
    print(f"Sampling rate: {fs} Hz")
    print(f"Duur: {raw.times[-1]/3600:.2f} uur")

    # EEG
    eeg = raw.get_data(picks=eeg_channel)[0]

    # Accelerometer
    dx = raw.get_data(picks=accel_channels[0])[0]
    dy = raw.get_data(picks=accel_channels[1])[0]
    dz = raw.get_data(picks=accel_channels[2])[0]

    # EMG (optioneel)
    emg = raw.get_data(picks=emg_channel)[0] if emg_channel else np.zeros_like(eeg)

    # Slaapstadia — zie opmerking hieronder
    n_epochs    = int(raw.times[-1] / 30)
    stage_epoch = [2] * n_epochs  

    return {
        "subject_id":  subject_id,
        "night_id":    night_id,
        "fs":          int(fs),
        "eeg":         eeg,
        "emg":         emg,
        "dx":          dx,
        "dy":          dy,
        "dz":          dz,
        "stage_epoch": stage_epoch,
    }

KeyboardInterrupt: 

Gebruik je het zo:
pythonfrom microarousal_pipeline import run_one_night

night = load_edf(
    edf_path   = "data/sub001_night1.edf",
    subject_id = "sub_001",
    night_id   = "night_1",
    eeg_channel = "EEG Fpz-Cz",   # pas aan op jouw kanaalnaam
)

df = run_one_night(night)

In [1]:
"""
Microarousal Detection Pipeline
================================
Volledig unsupervised detectie van kortdurende EEG-activaties (kandidaat-microarousals)
via Morlet CWT ridge-analyse, gevolgd door UMAP + HDBSCAN clustering.

Gebruik:
    from microarousal_pipeline import run_all_nights, cluster_events, plot_umap

    # Stap 1: verwerk alle nachten
    master = run_all_nights(night_list, output_dir="output/")

    # Stap 2: cluster alle events
    master = cluster_events(master)

    # Stap 3: visualiseer
    plot_umap(master)
    describe_clusters(master)
"""

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import butter, filtfilt, iirnotch
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ─────────────────────────────────────────────
# CONFIG — pas hier aan, nergens anders
# ─────────────────────────────────────────────

FS_DEFAULT       = 256       # sampling rate (Hz)
EPOCH_LEN        = 30        # seconden per slaapepoch
FREQ_MIN         = 1.0       # Hz — ondergrens EEG-ridge
FREQ_MAX         = 30.0      # Hz — bovengrens EEG-ridge (boven = EMG)
FREQ_EMG_MIN     = 70.0      # Hz — ondergrens EMG-proxy
N_FREQS          = 60        # aantal frequentiestappen voor CWT
MIN_EVENT_SEC    = 1.0       # minimale eventduur
MAX_EVENT_SEC    = 20.0      # maximale eventduur
MERGE_GAP_SEC    = 1.0       # merge events met gap kleiner dan dit
BASELINE_SEC     = 60.0      # lokale baseline venster (seconden)
FREQ_SHIFT_THR   = 3.0       # Hz boven baseline voor kandidaat-event

# slaapstadia die meegenomen worden (pas aan op jouw labels)
SLEEP_STAGES_INCLUDE = {0, 1, 2, 3, 5}


# ─────────────────────────────────────────────
# 1. MORLET CWT  (identiek aan Scoring Hero)
# ─────────────────────────────────────────────

def compute_morlet_tf(signal, srate, freqs, n_cycles=None, L2normalize=False):
    """
    Compute time-frequency power via Morlet wavelet convolution (FFT-based).
    Identiek aan compute_morlet_tf.py van Scoring Hero.
    Returns power: (n_freqs, n_samples)
    """
    freqs = np.asarray(freqs)
    if n_cycles is None:
        n_cycles_arr = np.maximum(3.0, freqs / 2.0)
    elif np.isscalar(n_cycles):
        n_cycles_arr = np.full(len(freqs), float(n_cycles))
    else:
        n_cycles_arr = np.asarray(n_cycles, dtype=float)

    signal = signal - np.mean(signal)
    signal_fft = np.fft.fft(signal)
    fft_freqs  = np.fft.fftfreq(len(signal), d=1.0 / srate)
    power      = np.empty((len(freqs), len(signal)), dtype=np.float64)

    for i, freq in enumerate(freqs):
        sigma_f     = freq / n_cycles_arr[i]
        wavelet_fft = np.exp(-0.5 * ((fft_freqs - freq) / sigma_f) ** 2)
        if L2normalize:
            wavelet_fft /= np.sqrt(np.sum(wavelet_fft ** 2))
        analytic  = np.fft.ifft(signal_fft * wavelet_fft)
        power[i]  = np.abs(analytic) ** 2

    return power


# ─────────────────────────────────────────────
# 2. PREPROCESSING
# ─────────────────────────────────────────────

def preprocess_eeg(raw, fs):
    """Bandpass 0.3–35 Hz + notch 50 Hz + DC removal."""
    sig = raw - np.mean(raw)
    b, a = butter(4, [0.3 / (fs/2), 35.0 / (fs/2)], btype='band')
    sig = filtfilt(b, a, sig)
    b, a = iirnotch(50.0 / (fs/2), Q=30)
    sig = filtfilt(b, a, sig)
    return sig


def movement_magnitude(dx, dy, dz):
    return np.sqrt(dx**2 + dy**2 + dz**2)


def stage_per_sample(stage_epoch, n_samples, fs, epoch_len=EPOCH_LEN):
    """Zet epoch-labels om naar sample-level array."""
    stages = np.zeros(n_samples, dtype=int)
    for i, s in enumerate(stage_epoch):
        start = int(i * epoch_len * fs)
        end   = int((i + 1) * epoch_len * fs)
        stages[start:min(end, n_samples)] = s
    return stages


# ─────────────────────────────────────────────
# 3. RIDGE EXTRACTIE
# ─────────────────────────────────────────────

def extract_ridge(eeg_clean, fs, freqs_eeg, freqs_full=None):
    """
    Berekent power matrix en trekt de ridge eruit.

    Returns
    -------
    power_eeg  : (n_freqs_eeg, n_samples)
    ridge_freq : (n_samples,)  — dominante freq per sample (1-30 Hz)
    ridge_pwr  : (n_samples,)  — power op de ridge
    emg_proxy  : (n_samples,)  — gemiddelde power 70-120 Hz als EMG-proxy
    """
    # EEG-bereik (1–30 Hz) — voor ridge en event-detectie
    power_eeg   = compute_morlet_tf(eeg_clean, fs, freqs_eeg)
    ridge_idx   = np.argmax(power_eeg, axis=0)
    ridge_freq  = freqs_eeg[ridge_idx]
    ridge_pwr   = power_eeg[ridge_idx, np.arange(power_eeg.shape[1])]

    # EMG-proxy: 70–120 Hz
    if freqs_full is not None:
        power_full  = compute_morlet_tf(eeg_clean, fs, freqs_full)
        gamma_mask  = freqs_full >= FREQ_EMG_MIN
        emg_proxy   = np.mean(power_full[gamma_mask, :], axis=0)
    else:
        # schat EMG-proxy vanuit de hoge kant van het 1-30 Hz bereik
        # (minder nauwkeurig maar sneller)
        emg_proxy = np.zeros(len(eeg_clean))

    return power_eeg, ridge_freq, ridge_pwr, emg_proxy


# ─────────────────────────────────────────────
# 4. BASELINE & ACTIVATIESCORE
# ─────────────────────────────────────────────

def rolling_median(arr, window):
    """Snelle rolling mediaan via stride tricks."""
    result = np.empty_like(arr)
    for i in range(len(arr)):
        start = max(0, i - window)
        result[i] = np.median(arr[start:i]) if i > 0 else arr[0]
    return result


def compute_freq_shift(ridge_freq, fs, baseline_sec=BASELINE_SEC):
    """
    Hoe ver springt de ridge omhoog t.o.v. de lokale baseline?
    Positief = activatie (snellere EEG), negatief = vertraging.
    """
    window    = int(baseline_sec * fs)
    baseline  = rolling_median(ridge_freq, window)
    return ridge_freq - baseline, baseline


# ─────────────────────────────────────────────
# 5. KANDIDAAT-EVENT DETECTIE
# ─────────────────────────────────────────────

def detect_events(ridge_freq, ridge_pwr, freq_shift, stages, fs,
                  threshold=FREQ_SHIFT_THR,
                  min_dur=MIN_EVENT_SEC,
                  max_dur=MAX_EVENT_SEC,
                  merge_gap=MERGE_GAP_SEC):
    """
    Detecteert kandidaat-events op basis van ridge-stijging.
    Geen klinische drempel — puur signaalgebaseerd.
    """
    active = freq_shift > threshold

    # verwijder samples buiten slaap
    in_sleep = np.isin(stages, list(SLEEP_STAGES_INCLUDE))
    active   = active & in_sleep

    min_s  = int(min_dur  * fs)
    max_s  = int(max_dur  * fs)
    merge_s = int(merge_gap * fs)

    # groepeer opeenvolgende actieve samples
    raw_events = []
    in_ev, start = False, 0
    for i, flag in enumerate(active):
        if flag and not in_ev:
            start, in_ev = i, True
        elif not flag and in_ev:
            in_ev = False
            if min_s <= (i - start) <= max_s:
                raw_events.append([start, i])

    # merge events dicht bij elkaar
    merged = []
    for ev in raw_events:
        if merged and (ev[0] - merged[-1][1]) < merge_s:
            merged[-1][1] = ev[1]
        else:
            merged.append(ev)

    # filter op duur na merge
    events = []
    for s, e in merged:
        dur = e - s
        if min_s <= dur <= max_s:
            events.append({
                "start_sample": s,
                "end_sample":   e,
                "start_sec":    s / fs,
                "end_sec":      e / fs,
                "duration":     dur / fs,
                "stage":        int(np.median(stages[s:e])),
                "mean_ridge_freq": float(np.mean(ridge_freq[s:e])),
                "peak_ridge_freq": float(np.max(ridge_freq[s:e])),
                "freq_shift":      float(np.mean(freq_shift[s:e])),
                "peak_power":      float(np.max(ridge_pwr[s:e])),
                "mean_power":      float(np.mean(ridge_pwr[s:e])),
            })

    return events


# ─────────────────────────────────────────────
# 6. FEATURES PER EVENT UIT POWER MATRIX
# ─────────────────────────────────────────────

def bandpower_from_matrix(power_matrix, freqs, lo, hi):
    """Gemiddelde power in frequentieband [lo, hi] over een tijdvenster."""
    mask = (freqs >= lo) & (freqs <= hi)
    return float(np.mean(power_matrix[mask, :]))


def extract_event_features(event, power_eeg, freqs_eeg, emg_proxy, ridge_freq,
                            fs, n_samples_total):
    """
    Berekent rijke feature-set per event uit de power matrix.
    Dit is de input voor clustering.
    """
    s = event["start_sample"]
    e = event["end_sample"]
    s = max(0, s)
    e = min(n_samples_total, e)

    pmat = power_eeg[:, s:e]

    # band powers
    delta  = bandpower_from_matrix(pmat, freqs_eeg, 0.5,  4.0)
    theta  = bandpower_from_matrix(pmat, freqs_eeg, 4.0,  8.0)
    alpha  = bandpower_from_matrix(pmat, freqs_eeg, 8.0, 12.0)
    sigma  = bandpower_from_matrix(pmat, freqs_eeg, 12.0, 16.0)  # spindle-range
    beta   = bandpower_from_matrix(pmat, freqs_eeg, 16.0, 30.0)
    total  = delta + theta + alpha + sigma + beta + 1e-10

    fast_slow = (alpha + beta) / (delta + theta + 1e-10)

    # ridge dynamiek
    ridge_segment = ridge_freq[s:e]
    ridge_slope   = float(np.polyfit(np.arange(len(ridge_segment)),
                                     ridge_segment, 1)[0]) if len(ridge_segment) > 2 else 0.0

    # EMG proxy
    emg_mean = float(np.mean(emg_proxy[s:e])) if emg_proxy is not None else 0.0

    return {
        **event,
        # band powers (genormaliseerd)
        "pow_delta":      delta  / total,
        "pow_theta":      theta  / total,
        "pow_alpha":      alpha  / total,
        "pow_sigma":      sigma  / total,
        "pow_beta":       beta   / total,
        # ratios
        "fast_slow_ratio": fast_slow,
        "beta_frac":       beta / total,
        # ridge kenmerken
        "ridge_slope":    ridge_slope,   # positief = freq stijgt tijdens event
        # EMG proxy (artefact indicator)
        "emg_proxy":      emg_mean,
    }


# ─────────────────────────────────────────────
# 7. ARTIFACT LABELING
# ─────────────────────────────────────────────

def label_artifact(row, emg_pct=90, acc_pct=90, emg_vals=None, acc_vals=None):
    """
    Markeer events als mogelijk artifact — gooit ze NIET weg,
    want clustering beslist zelf wat artifact-clusters zijn.
    """
    emg_thr = np.percentile(emg_vals, emg_pct) if emg_vals is not None else np.inf
    acc_thr = np.percentile(acc_vals, acc_pct) if acc_vals is not None else np.inf

    if row.get("emg_proxy", 0) > emg_thr and row.get("acc_mean", 0) > acc_thr:
        return "movement"
    if row.get("emg_proxy", 0) > emg_thr:
        return "emg_dominant"
    if row.get("acc_mean", 0) > acc_thr:
        return "movement_dominant"
    return "clean"


# ─────────────────────────────────────────────
# 8. PIPELINE PER NACHT
# ─────────────────────────────────────────────

def run_one_night(night: dict, compute_emg_proxy: bool = True) -> pd.DataFrame:
    """
    Verwerkt één nacht van begin tot eind.

    Parameters
    ----------
    night : dict met sleutels:
        subject_id, night_id, fs,
        eeg (raw), dx, dy, dz,
        stage_epoch (lijst van slaapstadia per 30s-epoch),
        optioneel: emg (raw)

    Returns
    -------
    DataFrame met één rij per kandidaat-event, inclusief alle features.
    """
    fs  = night.get("fs", FS_DEFAULT)
    eeg = preprocess_eeg(night["eeg"], fs)
    acc = movement_magnitude(night["dx"], night["dy"], night["dz"])

    stages_sample = stage_per_sample(night["stage_epoch"], len(eeg), fs)

    # frequentievectoren
    freqs_eeg  = np.linspace(FREQ_MIN, FREQ_MAX, N_FREQS)
    freqs_full = np.linspace(FREQ_MIN, 120.0, 200) if compute_emg_proxy else None

    print(f"  → CWT berekenen ({night['subject_id']} / {night['night_id']})...")
    power_eeg, ridge_freq, ridge_pwr, emg_proxy = extract_ridge(
        eeg, fs, freqs_eeg, freqs_full
    )

    freq_shift, baseline = compute_freq_shift(ridge_freq, fs)

    print(f"  → Events detecteren...")
    raw_events = detect_events(
        ridge_freq, ridge_pwr, freq_shift, stages_sample, fs
    )

    if not raw_events:
        print(f"  → Geen events gevonden.")
        return pd.DataFrame()

    print(f"  → {len(raw_events)} kandidaat-events gevonden, features berekenen...")

    # features per event
    records = []
    for ev in raw_events:
        s, e = ev["start_sample"], ev["end_sample"]
        acc_mean = float(np.mean(acc[s:e]))
        row = extract_event_features(
            ev, power_eeg, freqs_eeg, emg_proxy, ridge_freq, fs, len(eeg)
        )
        row["subject_id"] = night["subject_id"]
        row["night_id"]   = night["night_id"]
        row["acc_mean"]   = acc_mean
        records.append(row)

    df = pd.DataFrame(records)

    # artifact labeling (percentiel-gebaseerd per nacht)
    emg_vals = df["emg_proxy"].values
    acc_vals = df["acc_mean"].values
    df["artifact_label"] = df.apply(
        lambda r: label_artifact(r, emg_vals=emg_vals, acc_vals=acc_vals),
        axis=1
    )

    return df


# ─────────────────────────────────────────────
# 9. BATCH OVER ALLE NACHTEN
# ─────────────────────────────────────────────

def run_all_nights(night_list: list, output_dir: str = "output/") -> pd.DataFrame:
    """
    Verwerkt alle nachten en slaat per nacht een parquet op.
    Combineert alles in één master event table.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    all_dfs = []

    for i, night in enumerate(night_list):
        print(f"\n[{i+1}/{len(night_list)}] {night['subject_id']} — {night['night_id']}")
        try:
            df = run_one_night(night)
            if not df.empty:
                out_path = Path(output_dir) / f"{night['subject_id']}_{night['night_id']}.parquet"
                df.to_parquet(out_path, index=False)
                all_dfs.append(df)
                print(f"  → Opgeslagen: {out_path}")
        except Exception as ex:
            print(f"  ✗ Fout: {ex}")

    if not all_dfs:
        print("Geen events gevonden over alle nachten.")
        return pd.DataFrame()

    master = pd.concat(all_dfs, ignore_index=True)
    master.to_parquet(Path(output_dir) / "master_events.parquet", index=False)
    print(f"\n✓ Klaar — {len(master)} events, {master['subject_id'].nunique()} proefpersonen")
    return master


# ─────────────────────────────────────────────
# 10. UNSUPERVISED CLUSTERING
# ─────────────────────────────────────────────

CLUSTER_FEATURES = [
    "duration",
    "mean_ridge_freq",
    "freq_shift",
    "ridge_slope",
    "pow_delta",
    "pow_theta",
    "pow_alpha",
    "pow_sigma",
    "pow_beta",
    "fast_slow_ratio",
    "emg_proxy",
    "acc_mean",
    "stage",
]


def cluster_events(master: pd.DataFrame,
                   min_cluster_size: int = 50,
                   min_samples: int = 10,
                   umap_neighbors: int = 30,
                   umap_min_dist: float = 0.1) -> pd.DataFrame:
    """
    Dimensiereductie via UMAP + densiteitsclustering via HDBSCAN.
    Geen k opgeven — algoritme bepaalt zelf hoeveel clusters er zijn.
    Cluster -1 = noise (hoort nergens bij).
    """
    try:
        import umap
        import hdbscan
    except ImportError:
        raise ImportError("Installeer: pip install umap-learn hdbscan")

    cols = [c for c in CLUSTER_FEATURES if c in master.columns]
    X    = master[cols].dropna()

    # robuuste normalisatie (mediaan / MAD)
    med  = X.median()
    mad  = (X - med).abs().median() + 1e-10
    X_sc = (X - med) / (1.4826 * mad)

    print(f"UMAP: {len(X_sc)} events, {len(cols)} features...")
    reducer   = umap.UMAP(n_components=2, n_neighbors=umap_neighbors,
                          min_dist=umap_min_dist, random_state=42)
    embedding = reducer.fit_transform(X_sc)

    print("HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,
                                 min_samples=min_samples,
                                 prediction_data=True)
    labels    = clusterer.fit_predict(embedding)

    master = master.loc[X.index].copy()
    master["cluster"] = labels
    master["umap_x"]  = embedding[:, 0]
    master["umap_y"]  = embedding[:, 1]

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    print(f"✓ {n_clusters} clusters gevonden, {n_noise} noise-events ({100*n_noise/len(labels):.1f}%)")

    return master


# ─────────────────────────────────────────────
# 11. VISUALISATIE & QC
# ─────────────────────────────────────────────

def plot_umap(master: pd.DataFrame, save_path: str = "umap_clusters.png"):
    """UMAP scatter plot met cluster-kleuren."""
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm

    fig, ax = plt.subplots(figsize=(10, 8))

    noise = master["cluster"] == -1
    ax.scatter(master.loc[noise, "umap_x"], master.loc[noise, "umap_y"],
               c="lightgray", s=3, alpha=0.3, label=f"noise (n={noise.sum()})")

    clusters = sorted(master.loc[~noise, "cluster"].unique())
    colors   = cm.tab10(np.linspace(0, 1, len(clusters)))
    for cl, col in zip(clusters, colors):
        sub = master[master["cluster"] == cl]
        ax.scatter(sub["umap_x"], sub["umap_y"],
                   color=col, s=6, alpha=0.6,
                   label=f"cluster {cl} (n={len(sub)})")

    ax.legend(markerscale=2, fontsize=9)
    ax.set_title("UMAP — kandidaat-event clusters", fontsize=13)
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Opgeslagen: {save_path}")


def describe_clusters(master: pd.DataFrame) -> pd.DataFrame:
    """
    Mediaan per cluster voor alle features.
    Gebruik dit om te interpreteren wat elk cluster biologisch is.
    """
    cols = [c for c in CLUSTER_FEATURES + ["artifact_label"] if c in master.columns]
    summary = (master.groupby("cluster")[cols]
               .agg(["median", "std", "count"])
               .round(3))
    print(summary.to_string())
    return summary


def plot_qc_events(night: dict, events_df: pd.DataFrame,
                   n: int = 20, save_path: str = "qc_events.png"):
    """
    Plot n willekeurige events: EEG + ridge freq ter visuele inspectie.
    """
    import matplotlib.pyplot as plt

    fs  = night.get("fs", FS_DEFAULT)
    eeg = preprocess_eeg(night["eeg"], fs)

    sample = events_df.sample(min(n, len(events_df))).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 2, figsize=(16, len(sample) * 2.2))
    if len(sample) == 1:
        axes = [axes]

    freqs_eeg = np.linspace(FREQ_MIN, FREQ_MAX, N_FREQS)
    power_eeg, ridge_freq, _, _ = extract_ridge(eeg, fs, freqs_eeg)
    freq_shift, _ = compute_freq_shift(ridge_freq, fs)

    for i, (_, ev) in enumerate(sample.iterrows()):
        s   = int(ev["start_sec"] * fs)
        e   = int(ev["end_sec"]   * fs)
        pad = int(4 * fs)
        sl  = slice(max(0, s - pad), min(len(eeg), e + pad))

        # EEG
        axes[i][0].plot(eeg[sl], lw=0.5, color="steelblue")
        axes[i][0].axvspan(pad, pad + (e - s), alpha=0.25, color="orange")
        axes[i][0].set_title(
            f"EEG  |  dur={ev['duration']:.1f}s  stage={ev.get('stage','?')}  "
            f"cluster={ev.get('cluster','?')}  art={ev.get('artifact_label','?')}",
            fontsize=7)
        axes[i][0].axis("off")

        # Ridge frequentie
        axes[i][1].plot(ridge_freq[sl], lw=0.7, color="darkorange")
        axes[i][1].axvspan(pad, pad + (e - s), alpha=0.25, color="orange")
        axes[i][1].set_title("Ridge frequentie (Hz)", fontsize=7)
        axes[i][1].set_ylim(FREQ_MIN, FREQ_MAX)
        axes[i][1].axis("off")

    plt.suptitle("QC — willekeurige kandidaat-events", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Opgeslagen: {save_path}")


def qc_summary(master: pd.DataFrame) -> pd.DataFrame:
    """
    Kwaliteitscontrole per nacht:
    aantal events, events/uur, verdeling over stadia, artifact %.
    """
    def summarize(g):
        total_dur_hr = g["duration"].sum() / 3600 if "duration" in g.columns else 1
        return pd.Series({
            "n_events":       len(g),
            "events_per_hr":  round(len(g) / max(total_dur_hr, 0.1), 1),
            "mean_dur_sec":   round(g["duration"].mean(), 2),
            "pct_clean":      round(100 * (g.get("artifact_label","clean") == "clean").mean(), 1),
            "pct_noise":      round(100 * (g.get("cluster", 0) == -1).mean(), 1)
                              if "cluster" in g.columns else float("nan"),
        })

    summary = master.groupby(["subject_id", "night_id"]).apply(summarize).reset_index()
    print(summary.to_string())
    return summary


# ─────────────────────────────────────────────
# 12. KOPPELING AAN ANGSTSCORES
# ─────────────────────────────────────────────

def correlate_with_anxiety(master: pd.DataFrame,
                            questionnaire: pd.DataFrame,
                            anxiety_col: str = "anxiety_score") -> pd.DataFrame:
    """
    Berekent per proefpersoon het aantal events per cluster,
    en correleert dat met angstscores.

    Parameters
    ----------
    master        : event table met 'subject_id' en 'cluster'
    questionnaire : DataFrame met 'subject_id' en anxiety_col
    anxiety_col   : naam van de angstscorekolom

    Returns
    -------
    DataFrame met correlaties per cluster
    """
    from scipy.stats import spearmanr

    # events per cluster per subject
    pivot = (master.groupby(["subject_id", "cluster"])
             .size()
             .unstack(fill_value=0)
             .reset_index())

    merged = pivot.merge(questionnaire[["subject_id", anxiety_col]],
                         on="subject_id", how="inner")

    results = []
    for cl in [c for c in pivot.columns if c != "subject_id"]:
        r, p = spearmanr(merged[cl], merged[anxiety_col])
        results.append({
            "cluster":     cl,
            "spearman_r":  round(r, 3),
            "p_value":     round(p, 4),
            "n_subjects":  len(merged),
        })

    results_df = pd.DataFrame(results).sort_values("p_value")
    print(results_df.to_string())
    return results_df



In [ ]:
# ─────────────────────────────────────────────
# VOORBEELD GEBRUIK
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # ── simuleer één nacht (vervang dit door jouw data-loader) ──
    fs      = 256
    n_sec   = 3600          # 1 uur voor demo
    n_samp  = fs * n_sec
    rng     = np.random.default_rng(42)

    demo_night = {
        "subject_id":  "sub_001",
        "night_id":    "night_1",
        "fs":          fs,
        "eeg":         rng.standard_normal(n_samp) * 50,
        "dx":          rng.standard_normal(n_samp) * 0.01,
        "dy":          rng.standard_normal(n_samp) * 0.01,
        "dz":          rng.standard_normal(n_samp) * 0.01,
        # stadia: 2 = N2, 3 = N3, 5 = REM — vul in met jouw labels
        "stage_epoch": [2] * 60 + [3] * 30 + [5] * 30,
    }

    # stap 1: één nacht
    df = run_one_night(demo_night, compute_emg_proxy=False)
    print(df.head())

    # stap 2: alle nachten (vervang demo_night door jouw lijst)
    # master = run_all_nights([demo_night, ...], output_dir="output/")

    # stap 3: cluster
    # master = cluster_events(master)

    # stap 4: visualiseer
    # plot_umap(master)
    # describe_clusters(master)